# ML2+ML3 — Model Training (GPU variant)

**Variant of `02_training.ipynb` optimised for an NVIDIA GPU.**

- Drops LinearRegression (low signal, slow CV)
- Adds **XGBoost** with `device='cuda'` — GPU-accelerated gradient boosting
- Keeps **HistGradientBoosting** (sklearn CPU) as a baseline
- Reduces CV folds to 3 for faster iteration
- Auto-detects GPU and falls back to CPU gracefully

**Input:** `data/training_features.parquet`  
**Output:** `models/model.pkl`, `models/metadata.json`  
**Install:** `pip install xgboost>=2.0` (already in `requirements.txt`)

In [ ]:
import os
import json
import pickle
import subprocess
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# ── GPU detection ────────────────────────────────────────────────────────────
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, check=True)
    DEVICE = 'cuda'
    print('NVIDIA GPU detected — XGBoost will use CUDA')
    print(result.stdout.decode().split('\n')[0])  # first line of nvidia-smi
except Exception:
    DEVICE = 'cpu'
    print('No GPU detected — falling back to CPU for all models')

# ── Tune this if test R² is too low (try 0.90) ───────────────────────────────
TRAIN_RATIO = 0.85

FEATURE_COLS = [
    'hour', 'day_of_week', 'day_of_month', 'is_weekend', 'is_peak_hour',
    'fill_rate_1h_ago', 'fill_rate_24h_ago', 'fill_rate_7d_ago',
    'fill_rate_24h_avg', 'fill_rate_7d_avg', 'fill_rate_change_rate',
    'capacity_liters', 'type_id', 'density_km2',
]

print(f'XGBoost version : {xgb.__version__}')
print(f'TRAIN_RATIO     : {TRAIN_RATIO}')

## 1. Load & Inspect Dataset

In [ ]:
df = pd.read_parquet('data/training_features.parquet')
df = df.sort_values('measured_at').reset_index(drop=True)

if len(df) == 0:
    raise ValueError(
        'data/training_features.parquet is empty.\n'
        'Re-run 01_eda_feature_engineering.ipynb first.'
    )

print(f'Total rows   : {len(df):,}')
print(f'Date range   : {df.measured_at.min()} → {df.measured_at.max()}')
print(f'Containers   : {df.container_id.nunique():,}')
print(f'Target range : {df.target.min():.1f} – {df.target.max():.1f}')
df.head()

## 2. Temporal Train / Test Split

In [ ]:
split_idx = int(len(df) * TRAIN_RATIO)
df_train = df.iloc[:split_idx]
df_test  = df.iloc[split_idx:]

X_train, y_train = df_train[FEATURE_COLS].values, df_train['target'].values
X_test,  y_test  = df_test[FEATURE_COLS].values,  df_test['target'].values

print(f'Train : {len(df_train):,} rows  ({df_train.measured_at.min()} → {df_train.measured_at.max()})')
print(f'Test  : {len(df_test):,} rows  ({df_test.measured_at.min()} → {df_test.measured_at.max()})')

## 3. Cross-Validation Setup

3 folds instead of 5 — enough signal, meaningfully faster on large datasets.

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)
print(f'TimeSeriesSplit with {tscv.n_splits} folds')

## 4. XGBoost (GPU)

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=20,
    device=DEVICE,          # 'cuda' on GPU, 'cpu' fallback
    tree_method='hist',     # required for GPU; also fastest on CPU
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)

xgb_model.fit(X_train, y_train)

xgb_cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=tscv, scoring='r2')
xgb_cv_r2 = xgb_cv_scores.mean()

xgb_pred    = xgb_model.predict(X_test)
xgb_rmse    = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae     = mean_absolute_error(y_test, xgb_pred)
xgb_test_r2 = r2_score(y_test, xgb_pred)

print(f'XGBoost ({DEVICE.upper()})  CV R² (mean ± std): {xgb_cv_r2:.3f} ± {xgb_cv_scores.std():.3f}')
print(f'XGBoost ({DEVICE.upper()})  Test  RMSE={xgb_rmse:.2f}  MAE={xgb_mae:.2f}  R²={xgb_test_r2:.3f}')

## 5. HistGradientBoosting (CPU baseline)

In [ ]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=300,
    max_depth=6,
    min_samples_leaf=20,
    learning_rate=0.05,
    random_state=42,
)

hgb_model.fit(X_train, y_train)

hgb_cv_scores = cross_val_score(hgb_model, X_train, y_train, cv=tscv, scoring='r2')
hgb_cv_r2 = hgb_cv_scores.mean()

hgb_pred    = hgb_model.predict(X_test)
hgb_rmse    = np.sqrt(mean_squared_error(y_test, hgb_pred))
hgb_mae     = mean_absolute_error(y_test, hgb_pred)
hgb_test_r2 = r2_score(y_test, hgb_pred)

print(f'HistGradientBoosting (CPU)  CV R² (mean ± std): {hgb_cv_r2:.3f} ± {hgb_cv_scores.std():.3f}')
print(f'HistGradientBoosting (CPU)  Test  RMSE={hgb_rmse:.2f}  MAE={hgb_mae:.2f}  R²={hgb_test_r2:.3f}')

## 6. Model Comparison

In [ ]:
results = pd.DataFrame([
    {'Model': f'XGBoost ({DEVICE.upper()})', 'CV R²': xgb_cv_r2, 'Test R²': xgb_test_r2,
     'RMSE': xgb_rmse, 'MAE': xgb_mae},
    {'Model': 'HistGradientBoosting (CPU)', 'CV R²': hgb_cv_r2, 'Test R²': hgb_test_r2,
     'RMSE': hgb_rmse, 'MAE': hgb_mae},
]).set_index('Model')

print(f'\n=== Model comparison (TRAIN_RATIO={TRAIN_RATIO}) ===')
print(results.round(3))

print(f'\nTarget thresholds: RMSE < 10, MAE < 7, R² > 0.65')
for name, rmse, mae, r2 in [
    (f'XGBoost ({DEVICE.upper()})', xgb_rmse, xgb_mae, xgb_test_r2),
    ('HGB (CPU)',                   hgb_rmse, hgb_mae, hgb_test_r2),
]:
    ok = rmse < 10 and mae < 7 and r2 > 0.65
    print(f'  {name}: {"✅ PASS" if ok else "❌ FAIL"}')

In [ ]:
# CV score distributions
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(tscv.n_splits)
ax.plot(x, xgb_cv_scores, marker='o', label=f'XGBoost ({DEVICE.upper()})')
ax.plot(x, hgb_cv_scores, marker='^', label='HistGradientBoosting (CPU)')
ax.axhline(0.65, color='red', linestyle='--', label='Target R²=0.65')
ax.set_xlabel('CV fold')
ax.set_ylabel('R²')
ax.set_title(f'TimeSeriesSplit CV R² per fold  (TRAIN_RATIO={TRAIN_RATIO})')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Export Best Model

In [ ]:
candidates = [
    ('xgb', xgb_cv_r2, xgb_model, xgb_test_r2, xgb_rmse, xgb_mae),
    ('hgb', hgb_cv_r2, hgb_model, hgb_test_r2, hgb_rmse, hgb_mae),
]
best_name, best_cv_r2, best_model, best_test_r2, best_rmse, best_mae = max(
    candidates, key=lambda x: x[1]
)

os.makedirs('models', exist_ok=True)

with open('models/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

metadata = {
    'version':      f'{best_name}_v1.0',
    'trained_at':   datetime.now(timezone.utc).isoformat(),
    'feature_cols': FEATURE_COLS,
    'train_ratio':  TRAIN_RATIO,
    'device':       DEVICE,
    'cv_r2':        round(best_cv_r2,   4),
    'test_r2':      round(best_test_r2, 4),
    'test_rmse':    round(best_rmse,    4),
    'test_mae':     round(best_mae,     4),
    'model_type':   best_name,
    'train_rows':   len(df_train),
    'test_rows':    len(df_test),
    'n_features':   len(FEATURE_COLS),
}
with open('models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved  : models/model.pkl  ({best_name}_v1.0)')
print(f'Device : {DEVICE.upper()}')
print(f'CV R²  : {best_cv_r2:.3f}  |  Test R² : {best_test_r2:.3f}')
print(f'\nDone — proceed to 03_evaluation.ipynb')